# SNGP Embedding Extraction

Load an SNGP checkpoint, convert CIFAR-10 train/test plus OOD datasets into backbone embeddings, and save all arrays with labels. This notebook is the only GMM notebook that runs model inference.


In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import torch
import torchvision
import yaml
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / "src").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(f"Could not find repo root from {cwd}")

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data.cifar10 import get_cifar10_eval_transform
from src.data.cifar100 import get_cifar100_loaders
from src.data.svhn import get_svhn_loader
from src.models.sngp import SNGPResNetClassifier, laplace_predictive_probs

print("repo root:", REPO_ROOT)


repo root: /w/20252/wjcai/uq/manygp


In [2]:
CHECKPOINT_PATH = Path("/w/20251/wjcai/checkpoints_sngp/20260421_210700_0bb4464f/cifar10_sngp_epoch175_acc0.9323.pt")
FALLBACK_CONFIG_PATH = REPO_ROOT / "configs" / "cifar10_sngp.yaml"

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

cfg = checkpoint.get("config")
if cfg is None:
    with open(FALLBACK_CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)

checkpoint_id = CHECKPOINT_PATH.stem
print("device:", device)
print("checkpoint:", CHECKPOINT_PATH)
print("checkpoint id:", checkpoint_id)
print("checkpoint epoch:", checkpoint.get("epoch"))
print("stored val accuracy:", checkpoint.get("val_accuracy"))


device: cuda
checkpoint: /w/20251/wjcai/checkpoints_sngp/20260421_210700_0bb4464f/cifar10_sngp_epoch175_acc0.9323.pt
checkpoint id: cifar10_sngp_epoch175_acc0.9323
checkpoint epoch: 175
stored val accuracy: 0.9323


In [3]:
model_cfg = cfg["model"]
model = SNGPResNetClassifier(
    num_classes=model_cfg["num_classes"],
    width=model_cfg["width"],
    hidden_dim=model_cfg["hidden_dim"],
    spec_norm_bound=model_cfg["spec_norm_bound"],
    num_inducing=model_cfg["num_inducing"],
    ridge_penalty=model_cfg["ridge_penalty"],
    feature_scale=model_cfg["feature_scale"],
    gp_cov_momentum=model_cfg["gp_cov_momentum"],
    normalize_input=model_cfg["normalize_input"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("model loaded")


model loaded


In [4]:
data_cfg = cfg["data"]
data_root = data_cfg["root"]
batch_size = data_cfg.get("batch_size", 256)
num_workers = data_cfg.get("num_workers", 0)
svhn_split = cfg.get("ood", {}).get("svhn_split", "test")
id_normalization = cfg.get("ood", {}).get("id_normalization", "cifar10")

cifar10_eval_transform = get_cifar10_eval_transform()
cifar10_train_dataset = torchvision.datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=cifar10_eval_transform,
)
cifar10_test_dataset = torchvision.datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=cifar10_eval_transform,
)
cifar10_train_loader = DataLoader(
    cifar10_train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)
cifar10_test_loader = DataLoader(
    cifar10_test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

svhn_loader = get_svhn_loader(
    data_root=data_root,
    batch_size=batch_size,
    num_workers=num_workers,
    id_normalization=id_normalization,
    split=svhn_split,
)
_, cifar100_test_loader, _, cifar100_test_dataset = get_cifar100_loaders(
    data_root=data_root,
    batch_size=batch_size,
    num_workers=num_workers,
    smoke_test=False,
)

print("cifar10 train:", len(cifar10_train_dataset))
print("cifar10 test:", len(cifar10_test_dataset))
print("svhn split:", svhn_split, "size:", len(svhn_loader.dataset))
print("cifar100 test:", len(cifar100_test_dataset))


/w/20252/wjcai/uq/manygp/venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
/w/20252/wjcai/uq/manygp/venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


cifar10 train: 50000
cifar10 test: 10000
svhn split: test size: 26032
cifar100 test: 10000


In [5]:
@torch.no_grad()
def collect_embeddings(model, loader, device, num_mc_samples=10):
    model.eval()
    embeddings = []
    labels = []
    logits = []
    probs = []
    preds = []
    variances = []

    for images, batch_labels in loader:
        images = images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        batch_embeddings = model.encode(images)
        batch_logits, batch_variances = model(images, return_cov=True)
        batch_probs = laplace_predictive_probs(
            batch_logits,
            batch_variances,
            num_mc_samples=num_mc_samples,
        )

        embeddings.append(batch_embeddings.cpu())
        labels.append(batch_labels.cpu())
        logits.append(batch_logits.cpu())
        variances.append(batch_variances.cpu())
        probs.append(batch_probs.cpu())
        preds.append(batch_probs.argmax(dim=1).cpu())

    return {
        "embeddings": torch.cat(embeddings, dim=0).numpy(),
        "labels": torch.cat(labels, dim=0).numpy(),
        "logits": torch.cat(logits, dim=0).numpy(),
        "variances": torch.cat(variances, dim=0).numpy(),
        "probs": torch.cat(probs, dim=0).numpy(),
        "preds": torch.cat(preds, dim=0).numpy(),
    }


num_mc_samples = cfg.get("training", {}).get("num_mc_samples", 10)
split_loaders = {
    "cifar10_train": cifar10_train_loader,
    "cifar10_test": cifar10_test_loader,
    "svhn": svhn_loader,
    "cifar100_test": cifar100_test_loader,
}

split_outputs = {}
for split_name, loader in split_loaders.items():
    print("extracting:", split_name)
    split_outputs[split_name] = collect_embeddings(
        model=model,
        loader=loader,
        device=device,
        num_mc_samples=num_mc_samples,
    )
    print(" ", split_name, split_outputs[split_name]["embeddings"].shape)


extracting: cifar10_train
  cifar10_train (50000, 128)
extracting: cifar10_test
  cifar10_test (10000, 128)
extracting: svhn
  svhn (26032, 128)
extracting: cifar100_test
  cifar100_test (10000, 128)


In [6]:
cifar10_test_probs = split_outputs["cifar10_test"]["probs"]
cifar10_test_labels = split_outputs["cifar10_test"]["labels"]
cifar10_test_preds = split_outputs["cifar10_test"]["preds"]
cifar10_test_accuracy = np.mean(cifar10_test_preds == cifar10_test_labels)
cifar10_test_nll = -np.mean(
    np.log(np.clip(cifar10_test_probs[np.arange(len(cifar10_test_labels)), cifar10_test_labels], 1e-12, 1.0))
)

print(f"cifar10 test accuracy: {cifar10_test_accuracy * 100:.2f}%")
print(f"cifar10 test NLL: {cifar10_test_nll:.4f}")


cifar10 test accuracy: 93.18%
cifar10 test NLL: 0.3244


In [7]:
output_path = REPO_ROOT / "notebooks" / "gmm" / f"sngp_embeddings_{checkpoint_id}.npz"
np.savez_compressed(
    output_path,
    train_embeddings=split_outputs["cifar10_train"]["embeddings"],
    train_labels=split_outputs["cifar10_train"]["labels"],
    train_logits=split_outputs["cifar10_train"]["logits"],
    train_preds=split_outputs["cifar10_train"]["preds"],
    train_probs=split_outputs["cifar10_train"]["probs"],
    train_variances=split_outputs["cifar10_train"]["variances"],
    test_embeddings=split_outputs["cifar10_test"]["embeddings"],
    test_labels=split_outputs["cifar10_test"]["labels"],
    test_logits=split_outputs["cifar10_test"]["logits"],
    test_preds=split_outputs["cifar10_test"]["preds"],
    test_probs=split_outputs["cifar10_test"]["probs"],
    test_variances=split_outputs["cifar10_test"]["variances"],
    svhn_embeddings=split_outputs["svhn"]["embeddings"],
    svhn_labels=split_outputs["svhn"]["labels"],
    svhn_logits=split_outputs["svhn"]["logits"],
    svhn_preds=split_outputs["svhn"]["preds"],
    svhn_probs=split_outputs["svhn"]["probs"],
    svhn_variances=split_outputs["svhn"]["variances"],
    cifar100_test_embeddings=split_outputs["cifar100_test"]["embeddings"],
    cifar100_test_labels=split_outputs["cifar100_test"]["labels"],
    cifar100_test_logits=split_outputs["cifar100_test"]["logits"],
    cifar100_test_preds=split_outputs["cifar100_test"]["preds"],
    cifar100_test_probs=split_outputs["cifar100_test"]["probs"],
    cifar100_test_variances=split_outputs["cifar100_test"]["variances"],
    classes=np.array(cifar10_test_dataset.classes),
    cifar10_classes=np.array(cifar10_test_dataset.classes),
    cifar100_classes=np.array(cifar100_test_dataset.classes),
    svhn_classes=np.array([str(i) for i in range(10)]),
    checkpoint_id=checkpoint_id,
    checkpoint_path=str(CHECKPOINT_PATH),
    checkpoint_epoch=checkpoint.get("epoch", -1),
    checkpoint_val_accuracy=checkpoint.get("val_accuracy", np.nan),
    num_mc_samples=num_mc_samples,
    svhn_split=svhn_split,
    id_normalization=id_normalization,
    cifar10_test_accuracy=cifar10_test_accuracy,
    cifar10_test_nll=cifar10_test_nll,
)
print("saved:", output_path)


saved: /w/20252/wjcai/uq/manygp/notebooks/gmm/sngp_embeddings_cifar10_sngp_epoch175_acc0.9323.npz
